In [3]:
import xarray as xr

# Open a NetCDF file
ds = xr.open_dataset('selected_data\cropharvest_Kenya_maize_test.nc')

# Inspect data
print(ds)

<xarray.Dataset>
Dimensions:         (identifier: 898, DEM-D1: 2, S1-D1: 12, S1-D2: 2, S2-D1: 12, S2-D2: 11, VI-D1: 12, VI-D2: 1, view_available: 5, dim_target: 1, weather-D1: 12, weather-D2: 2)
Coordinates:
  * identifier      (identifier) int32 4705 4684 3952 4122 ... 4354 4474 4760
Dimensions without coordinates: DEM-D1, S1-D1, S1-D2, S2-D1, S2-D2, VI-D1, VI-D2, view_available, dim_target, weather-D1, weather-D2
Data variables:
    DEM             (identifier, DEM-D1) float32 ...
    S1              (identifier, S1-D1, S1-D2) float32 ...
    S2              (identifier, S2-D1, S2-D2) float32 ...
    VI              (identifier, VI-D1, VI-D2) float32 ...
    inverted_ident  (identifier, view_available) int32 ...
    target          (identifier, dim_target) int32 ...
    train_mask      (identifier) int32 ...
    weather         (identifier, weather-D1, weather-D2) float32 ...
Attributes:
    view_names:      ['S2', 'S1', 'weather', 'DEM', 'VI']
    target_names:    ['negative', 'posi

In [25]:
import torch
import numpy as np
targets = ds['target'].values
print(targets.dtype)
targets_tensor = torch.from_numpy(targets).to(torch.long)
print(targets_tensor.dtype)
print(targets_tensor.shape)
z = np.zeros(898)
print(list(zip(z, targets_tensor))[:5])
targets_numpy = targets_tensor.numpy()
print(targets_numpy.dtype)

int32
torch.int64
torch.Size([898, 1])
[(0.0, tensor([1])), (0.0, tensor([1])), (0.0, tensor([1])), (0.0, tensor([1])), (0.0, tensor([1]))]
int64


In [2]:
import yaml
import argparse
import os
import sys
import time
import gc
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.utils import class_weight

In [3]:
from src.training.utils import preprocess_views
from src.training.learn_pipeline import MultiFusion_train, assign_multifusion_name
from src.datasets.views_structure import DataViews, load_structure
from src.datasets.utils import _to_loader